# 07 - Final Integration: Tổng hợp Pipeline & Business Insights

**Môn học:** DATA ANALYSIS FOR BUSINESS ENVIRONMENT (71DAEE10012) – Sem1/2026-2027

**Đề tài:** Phân tích dữ liệu doanh thu và Phân cụm hành vi khách hàng trong lĩnh vực bán lẻ

**Dataset:** Online Retail Dataset – UCI Machine Learning Repository (~541.000 giao dịch)

---

**Mục đích notebook này:**
1. Chạy lại toàn bộ pipeline end-to-end từ Raw Data đến Customer Profiling.
2. Tổng hợp kết quả phân tích từ tất cả các phase.
3. Trình bày Business Insights dựa trên bằng chứng dữ liệu.
4. Nêu rõ Limitations của nghiên cứu.

**Pipeline tổng quan:**
```
Online Retail (.xlsx)
    → Data Import (Phase 1)
    → Data Cleaning (Phase 3)
    → Revenue Analysis (Phase 4)
    → Visualization (Phase 5)
    → Statistical Analysis (Phase 6)
    → RFM (Phase 7)
    → K-Means Clustering (Phase 8)
    → Cluster Profiling & Membership Tiers (Phase 9)
    → Business Insights & Technical Report (Phase 10)
```

## 0. Chuẩn bị thư viện & Nạp module

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập đường dẫn project root
current_dir = Path.cwd().resolve()
PROJECT_ROOT = current_dir.parent if current_dir.name == "notebooks" else current_dir
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

pd.options.display.float_format = "{:.2f}".format
pd.options.display.max_columns = 15
print(f"Project root: {PROJECT_ROOT}")

---
## PHẦN 1: TỔNG QUAN DỮ LIỆU VÀ QUY TRÌNH LÀM SẠCH

**Trả lời câu hỏi:** Dữ liệu thô gồm những gì? Đã xử lý như thế nào?

In [ ]:
# Nạp dữ liệu đã làm sạch (kết quả Phase 3)
df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "retail_cleaned.csv")
invoice_df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "invoice_data.csv")

print("=" * 60)
print("TỔNG QUAN DỮ LIỆU SAU KHI LÀM SẠCH")
print("=" * 60)
print(f"Số dòng giao dịch sạch     : {len(df):,}")
print(f"Số cột                     : {df.shape[1]}")
print(f"Số hóa đơn duy nhất        : {df['InvoiceNo'].nunique():,}")
print(f"Số khách hàng có CustomerID : {df['CustomerID'].dropna().nunique():,}")
print(f"Số quốc gia                : {df['Country'].nunique()}")
print(f"Khoảng thời gian           : {df['InvoiceDate'].min()} → {df['InvoiceDate'].max()}")
print(f"\nTỷ lệ đơn hủy (IsCancelled): {df['IsCancelled'].mean()*100:.2f}%")
print(f"Tổng doanh thu thuần (Net)  : £{df['Revenue'].sum():,.2f}")

### Tóm tắt quy trình Data Cleaning (Phase 3)

| Bước | Xử lý | Lý do |
|:-----|:-------|:------|
| 1 | Xóa 5,268 dòng trùng lặp hoàn toàn | Lỗi hệ thống ghi nhận đúp |
| 2 | Gắn cờ `IsCancelled` cho InvoiceNo bắt đầu bằng `C` | Phân biệt đơn mua và đơn hủy |
| 3 | Loại bỏ dòng có UnitPrice ≤ 0 | Bút toán kế toán, hàng tặng kèm |
| 4 | Loại bỏ dòng Quantity < 0 KHÔNG phải đơn hủy | Điều chỉnh kho hư hỏng/mất mát |
| 5 | Tạo biến `Revenue = Quantity × UnitPrice` | Biến phái sinh cốt lõi |
| 6 | **KHÔNG xóa** dòng thiếu CustomerID | Giữ doanh thu của khách vãng lai (~25% dữ liệu) |

---
## PHẦN 2: KẾT QUẢ PHÂN TÍCH DOANH THU (Q1)

**Câu hỏi Q1:** Hoạt động bán hàng thay đổi như thế nào theo thời gian, sản phẩm, quốc gia?

In [ ]:
# Nạp các bảng kết quả Phase 4
monthly_df = pd.read_csv(PROJECT_ROOT / "outputs" / "tables" / "monthly_revenue.csv")
product_df = pd.read_csv(PROJECT_ROOT / "outputs" / "tables" / "top_products_revenue.csv")
country_df = pd.read_csv(PROJECT_ROOT / "outputs" / "tables" / "country_revenue.csv")
invoice_stats = pd.read_csv(PROJECT_ROOT / "outputs" / "tables" / "invoice_value_summary.csv")

print("=" * 60)
print("A. DOANH THU THEO THỜI GIAN (Monthly Net Revenue)")
print("=" * 60)
print(monthly_df[["YearMonth", "NetRevenue"]].to_string(index=False))
print(f"\nTháng doanh thu cao nhất : {monthly_df.loc[monthly_df['NetRevenue'].idxmax(), 'YearMonth']}")
print(f"Giá trị                 : £{monthly_df['NetRevenue'].max():,.2f}")

In [ ]:
print("=" * 60)
print("B. TOP 10 SẢN PHẨM THEO DOANH THU")
print("=" * 60)
print(product_df[["StockCode", "Description", "TotalRevenue"]].to_string(index=False))

In [ ]:
print("=" * 60)
print("C. DOANH THU THEO QUỐC GIA (Top 10)")
print("=" * 60)
print(country_df[["Country", "TotalRevenue", "PercentageShare"]].to_string(index=False))
print(f"\nThị trường UK chiếm: {country_df.loc[0, 'PercentageShare']:.1f}% tổng doanh thu")

### Observations từ Q1:

1. **Theo thời gian:** Doanh thu tăng trưởng mạnh vào quý 4 (tháng 9-11/2011), đạt đỉnh £1.46M vào tháng 11/2011. Tháng 12/2011 giảm mạnh do chỉ thu thập dữ liệu đến ngày 09/12.
2. **Theo sản phẩm:** Các mặt hàng gia dụng, trang trí tiệc và quà tặng lưu niệm là trụ cột doanh thu.
3. **Theo quốc gia:** United Kingdom chiếm ~84% tổng doanh thu, phản ánh mô hình kinh doanh tập trung thị trường nội địa. Thị trường quốc tế (Netherlands, Ireland, Germany) chiếm tỷ trọng nhỏ.

> **Lưu ý:** Revenue ≠ Profit. Dataset không cung cấp chi phí (COGS) để tính lợi nhuận.

---
## PHẦN 3: KẾT QUẢ KIỂM ĐỊNH THỐNG KÊ (Q4)

**Câu hỏi Q4:** Một số khác biệt quan sát được có ý nghĩa thống kê hay không?

In [ ]:
stat_df = pd.read_csv(PROJECT_ROOT / "outputs" / "tables" / "statistical_tests_summary.csv")

print("=" * 60)
print("BẢNG TỔNG HỢP KẾT QUẢ KIỂM ĐỊNH THỐNG KÊ (Phase 6)")
print("=" * 60)
for i, row in stat_df.iterrows():
    print(f"\n--- Kiểm định {i+1} ---")
    print(f"  Câu hỏi      : {row['Câu hỏi nghiên cứu']}")
    print(f"  Kiểm định     : {row['Kiểm định sử dụng']}")
    print(f"  Lý do chọn    : {row['Lý do chọn test']}")
    print(f"  Thống kê      : {row['Thống kê kiểm định']}")
    print(f"  p-value       : {row['p-value']}")
    print(f"  Kết luận      : {row['Kết luận']}")

### Diễn giải chuẩn thống kê:

**Kiểm định 1 — Giá trị đơn hàng Weekday vs Weekend:**
- Dùng **Mann-Whitney U Test** (phi tham số) vì phân phối Invoice Value bị lệch phải rất mạnh, vi phạm giả định phân phối chuẩn của t-test.
- Kết quả: p-value < 0.05 → Bác bỏ H₀. Có đủ bằng chứng thống kê cho thấy phân phối giá trị đơn hàng giữa Weekday và Weekend khác nhau.
- **Lưu ý:** Bác bỏ H₀ KHÔNG có nghĩa là sự khác biệt lớn, chỉ có nghĩa là nó có ý nghĩa thống kê ở mức α = 0.05.

**Kiểm định 2 — Country vs Cancellation:**
- Dùng **Chi-Square Test of Independence** vì cả 2 biến đều là biến định tính (Categorical).
- Kết quả: p-value < 0.05 → Bác bỏ H₀. Có mối liên hệ có ý nghĩa thống kê giữa quốc gia và việc hủy đơn. Germany có tỷ lệ hủy đơn cao vượt trội (~24.2%).

---
## PHẦN 4: MÔ HÌNH HÓA HÀNH VI KHÁCH HÀNG - RFM (Q2)

**Câu hỏi Q2:** Hành vi mua hàng của từng khách hàng có thể được mô tả như thế nào?

In [ ]:
rfm_df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "rfm_data.csv")
rfm_summary = pd.read_csv(PROJECT_ROOT / "outputs" / "tables" / "rfm_summary.csv", index_col=0)

print("=" * 60)
print("BẢNG RFM - MÔ TẢ HÀNH VI KHÁCH HÀNG (Phase 7)")
print("=" * 60)
print(f"Số khách hàng hợp lệ: {len(rfm_df):,}")
print(f"\n--- Thống kê mô tả phân phối RFM ---")
print(rfm_summary.to_string())
print(f"\n--- 10 khách hàng đầu tiên ---")
rfm_df.head(10)

### Phân biệt quan trọng:
- **RFM** là bảng biểu diễn đặc trưng hành vi (Feature Representation), KHÔNG phải là phân cụm.
- **K-Means** là thuật toán phân cụm (Clustering Algorithm) sử dụng RFM làm đầu vào.

### Nhận xét phân phối RFM:
- **Recency:** Trung vị ~50 ngày, Skewness = 1.25 (lệch phải nhẹ).
- **Frequency:** Trung vị chỉ 2 đơn, Max 209 đơn, Skewness = 12.05 (lệch phải rất mạnh).
- **Monetary:** Trung vị £653, Max £279,489, Skewness = 21.59 (lệch phải cực đoan).

> **Hệ quả kỹ thuật:** Frequency và Monetary có phân phối lệch phải cực mạnh → Bắt buộc phải Log-transform + StandardScaler trước khi đưa vào K-Means.

---
## PHẦN 5: PHÂN CỤM K-MEANS & ĐÁNH GIÁ (Q3)

**Câu hỏi Q3:** Có thể phân khách hàng thành các nhóm có hành vi tương đồng hay không?

In [ ]:
eval_df = pd.read_csv(PROJECT_ROOT / "outputs" / "tables" / "kmeans_evaluation_metrics.csv")

print("=" * 60)
print("BẢNG ĐÁNH GIÁ K-MEANS TRÊN DẢI K = 2..7 (Phase 8)")
print("=" * 60)
print(eval_df.to_string(index=False))
print(f"\n→ K đã chọn: K = 3 (Điểm uốn Elbow rõ ràng nhất, Silhouette = 0.34)")

### Lý do chọn K = 3:

| Tiêu chí | K=2 | **K=3** | K=4 | K=5+ |
|:---------|:---:|:-------:|:---:|:----:|
| Inertia (WCSS) | 6,357 | **4,806** | 3,915 | < 3,400 |
| Silhouette Score | 0.4312 | **0.3400** | 0.3354 | < 0.31 |
| Điểm uốn Elbow | ✗ | **✓ (rõ nhất)** | ✓ (nhẹ) | ✗ |
| Ý nghĩa kinh doanh | Quá thô (2 nhóm) | **3 phân khúc rõ ràng** | Thêm nhóm VIP | Phân mảnh nhỏ |

**Quyết định:** Chọn K = 3 — cân bằng giữa chỉ số toán học và khả năng diễn giải kinh doanh.

---
## PHẦN 6: HỒ SƠ PHÂN KHÚC KHÁCH HÀNG & HẠNG THẺ HỘI VIÊN (Phase 9)

In [ ]:
segments_df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "customer_segments.csv")
tier_summary = pd.read_csv(PROJECT_ROOT / "outputs" / "tables" / "customer_segments_summary.csv")

print("=" * 60)
print("BẢNG HỒ SƠ PHÂN KHÚC KHÁCH HÀNG (MEMBERSHIP TIERS)")
print("=" * 60)
# Hiển thị bảng tóm tắt với các cột chính
display_cols = [
    "MembershipTier", "CustomerCount", "Customer_Pct",
    "Recency_Median", "Frequency_Median", "Monetary_Median",
    "Revenue_Share_Pct"
]
print(tier_summary[display_cols].to_string(index=False))

### Phân biệt quan trọng:
- **"K-Means tạo Cluster"**: Thuật toán chỉ phân 3 cụm toán học (Cluster 0, 1, 2).
- **"Nhóm nghiên cứu diễn giải Cluster"**: Dựa trên quan sát chỉ số R, F, M thực tế của từng cụm để ánh xạ sang tên hạng thẻ hội viên.

### Quy tắc ánh xạ:
- Sắp xếp cụm theo **Monetary_Mean tăng dần** (khách quan, không phụ thuộc vào thứ tự nhãn của thuật toán).
- Cụm chi tiêu thấp nhất → **Không Thẻ / Hạng Chuẩn**
- Cụm chi tiêu trung bình → **Thẻ Bạc (Silver)**
- Cụm chi tiêu cao nhất → **Thẻ Vàng - Kim Cương (Gold)**

---
## PHẦN 7: TỔNG HỢP BUSINESS INSIGHTS

Tất cả insight dưới đây đều **xuất phát trực tiếp từ kết quả phân tích**, không bịa đặt nguyên nhân mà dataset không hỗ trợ.

In [ ]:
print("=" * 70)
print("         TỔNG HỢP BUSINESS INSIGHTS")
print("=" * 70)

# Insight 1: Pareto Effect
gold = tier_summary[tier_summary["MembershipTier"].str.contains("Vàng")].iloc[0]
print(f"""
INSIGHT 1: QUY LUẬT PARETO (80/20)
─────────────────────────────────────
  Nhóm Thẻ Vàng - Kim Cương chiếm {gold['Customer_Pct']:.1f}% tổng khách hàng
  nhưng đóng góp {gold['Revenue_Share_Pct']:.1f}% tổng doanh thu (£{gold['TotalMonetary']:,.0f}).
  
  → Observation: Doanh thu tập trung cao độ vào nhóm nhỏ khách hàng.
  → Đây là pattern: {gold['Customer_Pct']:.0f}/{gold['Revenue_Share_Pct']:.0f} (gần quy luật 20/80).
""")

# Insight 2: Seasonality
print(f"""
INSIGHT 2: TÍNH MÙA VỤ DOANH THU
─────────────────────────────────────
  Doanh thu thuần quý 4 (tháng 9-11/2011): £{monthly_df[monthly_df['YearMonth'].isin(['2011-09','2011-10','2011-11'])]['NetRevenue'].sum():,.0f}
  Đỉnh cao nhất: tháng 11/2011 (£{monthly_df['NetRevenue'].max():,.0f})
  
  → Observation: Doanh thu tăng mạnh vào mùa mua sắm cuối năm (Q4).
  → Limitation: Dataset không chứa dữ liệu campaign marketing,
    nên KHÔNG THỂ khẳng định nguyên nhân tăng trưởng.
""")

# Insight 3: Geographic concentration
uk_share = country_df.loc[0, 'PercentageShare']
print(f"""
INSIGHT 3: TẬP TRUNG ĐỊA LÝ
─────────────────────────────────────
  United Kingdom chiếm {uk_share:.1f}% tổng doanh thu.
  Top 5 quốc gia chiếm {country_df.head(5)['PercentageShare'].sum():.1f}% tổng doanh thu.
  
  → Observation: Mô hình kinh doanh phụ thuộc gần như hoàn toàn vào thị trường nội địa.
""")

# Insight 4: Cancellation risk
standard = tier_summary[tier_summary["MembershipTier"].str.contains("Chuẩn")].iloc[0]
print(f"""
INSIGHT 4: NHÓM NGUY CƠ RỜI BỎ
─────────────────────────────────────
  Nhóm Không Thẻ / Hạng Chuẩn: {standard['Customer_Pct']:.1f}% khách hàng
  nhưng chỉ đóng góp {standard['Revenue_Share_Pct']:.1f}% doanh thu.
  Recency trung vị: {standard['Recency_Median']:.0f} ngày (đã >5 tháng không quay lại).
  Frequency trung vị: {standard['Frequency_Median']:.0f} đơn (hầu hết chỉ mua 1 lần).
  
  → Observation: Gần một nửa khách hàng có giá trị đóng góp rất thấp
    và có dấu hiệu ngừng hoạt động.
""")

# Insight 5: Statistical evidence
print(f"""
INSIGHT 5: BẰNG CHỨNG THỐNG KÊ
─────────────────────────────────────
  (a) Giá trị đơn hàng ngày Weekday khác biệt có ý nghĩa thống kê
      so với Weekend (Mann-Whitney U, p < 0.05).
  (b) Quốc gia có mối liên hệ có ý nghĩa thống kê với tỷ lệ hủy đơn
      (Chi-Square, p < 0.05). Đức có tỷ lệ hủy đơn ~24.2%.
""")

---
## PHẦN 8: BIỂU ĐỒ TỔNG HỢP (Minh họa key findings)

In [ ]:
from IPython.display import Image, display

figures_dir = PROJECT_ROOT / "outputs" / "figures"

print("Biểu đồ 1: Xu hướng doanh thu theo tháng")
display(Image(filename=str(figures_dir / "monthly_revenue_trend.png"), width=750))

In [ ]:
print("Biểu đồ 2: Top 10 sản phẩm đóng góp doanh thu")
display(Image(filename=str(figures_dir / "top_products_revenue.png"), width=750))

In [ ]:
print("Biểu đồ 3: Cơ cấu doanh thu theo quốc gia")
display(Image(filename=str(figures_dir / "country_revenue_breakdown.png"), width=750))

In [ ]:
print("Biểu đồ 4: Phân phối giá trị hóa đơn")
display(Image(filename=str(figures_dir / "invoice_value_distribution.png"), width=750))

In [ ]:
print("Biểu đồ 5: Elbow & Silhouette (Đánh giá K-Means)")
display(Image(filename=str(figures_dir / "kmeans_elbow_silhouette.png"), width=750))

In [ ]:
print("Biểu đồ 6: Hồ sơ phân khúc khách hàng (Membership Tiers)")
display(Image(filename=str(figures_dir / "customer_segments_profile.png"), width=750))

---
## PHẦN 9: LIMITATIONS (Giới hạn nghiên cứu)

Phần này nêu rõ những giới hạn mà nhóm nhận thức được, tránh suy luận vượt quá phạm vi dữ liệu.

| # | Giới hạn | Giải thích |
|:-:|:---------|:-----------|
| 1 | **Không có demographic data** | Dataset không chứa tuổi, giới tính, thu nhập → Không thể phân tích nhân khẩu học. |
| 2 | **Không có dữ liệu campaign marketing** | Không thể gán sự tăng trưởng doanh thu cho bất kỳ chiến dịch cụ thể nào. |
| 3 | **Không có chi phí (Cost/COGS)** | Revenue ≠ Profit. Không thể tính lợi nhuận ròng. |
| 4 | **RFM chỉ mô tả hành vi giao dịch** | RFM không nắm bắt sở thích sản phẩm, lý do mua hàng hay mức độ hài lòng. |
| 5 | **K-Means phụ thuộc feature selection & scaling** | Thay đổi phương pháp scaling hoặc thêm/bớt đặc trưng có thể cho kết quả khác. |
| 6 | **Cancellation cần xử lý cẩn thận** | Đơn hủy ảnh hưởng đến Monetary (Net Spend), cách xử lý khác nhau sẽ cho RFM khác nhau. |
| 7 | **Dữ liệu chỉ từ 12/2010 – 12/2011** | Kết quả chỉ phản ánh hành vi trong 1 năm, không thể ngoại suy sang giai đoạn khác. |
| 8 | **~25% giao dịch thiếu CustomerID** | Các phân tích ở cấp độ khách hàng (RFM, K-Means) chỉ dựa trên ~75% dữ liệu có mã khách hàng. |
| 9 | **Tên hạng thẻ là diễn giải của nhóm nghiên cứu** | K-Means chỉ tạo Cluster 0, 1, 2. Việc đặt tên "Vàng", "Bạc", "Chuẩn" là interpretation, không phải kết quả thuật toán. |

---
## PHẦN 10: PIPELINE TỔNG HỢP — CHẠY LẠI END-TO-END

Đoạn code dưới đây chạy lại toàn bộ pipeline từ đầu đến cuối, đảm bảo tính tái lập (Reproducibility).

In [ ]:
# =====================================================
# FULL PIPELINE — Chạy lại toàn bộ từ Raw Data đến kết quả cuối
# =====================================================
# LƯU Ý: Cell này mất ~2-3 phút do đọc file Excel lớn.
# Chỉ chạy khi cần tái tạo toàn bộ kết quả từ đầu.

RUN_FULL_PIPELINE = False  # Đặt = True nếu muốn chạy lại toàn bộ

if RUN_FULL_PIPELINE:
    print("=" * 60)
    print("PHASE 3: Data Cleaning")
    print("=" * 60)
    from src.data.cleaning import run_cleaning_pipeline
    df_clean, df_invoice = run_cleaning_pipeline()
    
    print("\n" + "=" * 60)
    print("PHASE 4: Revenue Analysis")
    print("=" * 60)
    from src.analysis.revenue_analysis import export_revenue_reports
    export_revenue_reports()
    
    print("\n" + "=" * 60)
    print("PHASE 5: Visualization")
    print("=" * 60)
    from src.visualization.plots import generate_all_revenue_visualizations
    generate_all_revenue_visualizations()
    
    print("\n" + "=" * 60)
    print("PHASE 6: Statistical Analysis")
    print("=" * 60)
    from src.analysis.statistics import run_all_statistical_tests
    run_all_statistical_tests()
    
    print("\n" + "=" * 60)
    print("PHASE 7: RFM")
    print("=" * 60)
    from src.modeling.rfm import export_rfm_pipeline
    export_rfm_pipeline()
    
    print("\n" + "=" * 60)
    print("PHASE 8: K-Means Evaluation")
    print("=" * 60)
    from src.modeling.kmeans import run_kmeans_evaluation_pipeline
    run_kmeans_evaluation_pipeline()
    
    print("\n" + "=" * 60)
    print("PHASE 9: Cluster Profiling")
    print("=" * 60)
    from src.analysis.customer_analysis import run_customer_profiling_pipeline
    run_customer_profiling_pipeline()
    
    print("\n" + "=" * 60)
    print("PIPELINE HOÀN TẤT THÀNH CÔNG")
    print("=" * 60)
else:
    print("Pipeline không được chạy lại (RUN_FULL_PIPELINE = False).")
    print("Đặt RUN_FULL_PIPELINE = True ở đầu cell nếu muốn tái tạo toàn bộ kết quả.")

---
## PHẦN 11: SƠ ĐỒ PIPELINE TỔNG THỂ

```
data/raw/Online Retail.xlsx
        │
        ▼
┌───────────────────────────────────────────────┐
│  src/data/load_data.py                        │
│  → Đọc file Excel, kiểm tra cột              │
└───────────────────┬───────────────────────────┘
                    ▼
┌───────────────────────────────────────────────┐
│  src/data/cleaning.py                         │
│  → Xóa trùng lặp, gắn IsCancelled            │
│  → Lọc dòng không hợp lệ                     │
│  → Tạo Revenue = Quantity × UnitPrice         │
│  → Xuất: retail_cleaned.csv, invoice_data.csv │
└───────────────────┬───────────────────────────┘
                    ▼
┌───────────────────────────────────────────────┐
│  src/analysis/revenue_analysis.py             │
│  → Doanh thu theo Tháng / Sản phẩm / Nước    │
│  → Phân phối Invoice Value                    │
│  → Xuất: outputs/tables/*.csv                 │
└───────────────────┬───────────────────────────┘
                    ▼
┌───────────────────────────────────────────────┐
│  src/visualization/plots.py                   │
│  → Line chart, Bar chart, Histogram + KDE     │
│  → Xuất: outputs/figures/*.png                │
└───────────────────┬───────────────────────────┘
                    ▼
┌───────────────────────────────────────────────┐
│  src/analysis/statistics.py                   │
│  → Mann-Whitney U Test (Invoice Value)        │
│  → Chi-Square Test (Country × Cancellation)   │
│  → Xuất: statistical_tests_summary.csv        │
└───────────────────┬───────────────────────────┘
                    ▼
┌───────────────────────────────────────────────┐
│  src/modeling/rfm.py                          │
│  → Tổng hợp RFM ở Customer Level             │
│  → Xuất: rfm_data.csv, rfm_summary.csv        │
└───────────────────┬───────────────────────────┘
                    ▼
┌───────────────────────────────────────────────┐
│  src/modeling/kmeans.py                       │
│  → Log-transform + StandardScaler             │
│  → K-Means (K=2..7), Elbow, Silhouette        │
│  → Chọn K=3, gán nhãn Cluster                │
│  → Xuất: kmeans_evaluation_metrics.csv        │
└───────────────────┬───────────────────────────┘
                    ▼
┌───────────────────────────────────────────────┐
│  src/analysis/customer_analysis.py            │
│  → Profile từng cụm (Mean, Median, % Share)   │
│  → Ánh xạ Hạng thẻ Hội viên                  │
│  → Xuất: customer_segments.csv                │
│           customer_segments_summary.csv        │
│           customer_segments_profile.png        │
└───────────────────┬───────────────────────────┘
                    ▼
            Business Insights
            & Technical Report
```

---
## PHẦN 12: XUẤT BÁO CÁO ĐỊNH DẠNG EXCEL & PDF

Hệ thống cung cấp module `src.utils.export` để tự động đóng gói toàn bộ bảng biểu phân tích sang file Excel (.xlsx) đa trang chuẩn Dashboard, đồng thời biên dịch Báo cáo Kỹ thuật sang file PDF (.pdf) hoàn chỉnh.

In [ ]:
from src.utils.export import export_all_reports

# Thực thi xuất toàn bộ báo cáo
generated_files = export_all_reports()

print('\nCác file đã xuất thành công trong thư mục outputs/reports/:')
for doc_type, file_path in generated_files.items():
    print(f'- [{doc_type.upper()}] {file_path.name} ({file_path.stat().st_size / 1024:.1f} KB)')


---

**© 2026 — Online Retail Data Analysis Project — Sem1/2026-2027**